In [1]:
import numpy as np
import pandas as pd

votes = pd.read_csv("../data/processed/votes.csv")
res = pd.read_csv("../data/processed/resolutions.csv", parse_dates=["date"])

M = votes.pivot(index="ms_code", columns="undl_id", values="vote")
print(M.shape)   # countries x resolutions

(202, 5694)


In [2]:
MIN_COMMON = 20   # ignore pairs with too few shared votes

def agreement_matrix(M):
    A = M.to_numpy()
    valid = (~np.isnan(A)).astype(float)
    common = valid @ valid.T                     # votes both countries cast
    agree = np.zeros_like(common)
    for v in (1.0, 0.0, -1.0):
        onehot = (A == v).astype(float)
        agree += onehot @ onehot.T               # votes where both chose v
    with np.errstate(invalid="ignore", divide="ignore"):
        S = agree / common
    S[common < MIN_COMMON] = np.nan
    S = pd.DataFrame(S, index=M.index, columns=M.index)
    C = pd.DataFrame(common, index=M.index, columns=M.index)
    return S, C

In [3]:
def window(M, start, end):
    ids = res.loc[(res["date"] >= start) & (res["date"] <= end), "undl_id"]
    sub = M[M.columns.intersection(ids)]
    return sub.dropna(how="all")                 # drop non-members

sub = window(M, "2000-01-01", "2025-12-31")
S, C = agreement_matrix(sub)
print(sub.shape)
S.loc["USA"].dropna().sort_values(ascending=False).head(10)

(195, 2232)


ms_code
USA    1.000000
ISR    0.828491
FSM    0.681561
MHL    0.581530
PLW    0.579434
CAN    0.522052
GBR    0.507886
AUS    0.461712
FRA    0.456963
HUN    0.427088
Name: USA, dtype: float64

In [4]:
S.loc["USA"].dropna().sort_values(ascending=False).head(10)

ms_code
USA    1.000000
ISR    0.828491
FSM    0.681561
MHL    0.581530
PLW    0.579434
CAN    0.522052
GBR    0.507886
AUS    0.461712
FRA    0.456963
HUN    0.427088
Name: USA, dtype: float64

In [5]:
sub = window(M, "2000-01-01", "2024-12-31")
S, C = agreement_matrix(sub)
S.loc["USA"].dropna().sort_values(ascending=False).head(10)

ms_code
USA    1.000000
ISR    0.833082
FSM    0.724845
MHL    0.631272
PLW    0.622593
CAN    0.555665
GBR    0.535767
AUS    0.490138
FRA    0.481006
HUN    0.445378
Name: USA, dtype: float64

In [6]:
  top = S.loc["USA"].drop("USA").dropna().sort_values(ascending=False).head(10)
  pd.DataFrame({"agreement": top.round(3), "shared_votes": C.loc["USA", top.index].astype(int)})

,agreement,shared_votes
ms_code,,
ISR,0.833,1989
FSM,0.725,1610
MHL,0.631,1714
PLW,0.623,1558
CAN,0.556,2030
GBR,0.536,2027
AUS,0.490,2028
FRA,0.481,2027
HUN,0.445,2023


In [7]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

names = pd.read_csv("../data/processed/countries.csv").set_index("ms_code")["name"]

def cluster_window(start, end, min_coverage=0.7, n_clusters=5):
    sub = window(M, start, end)
    coverage = sub.notna().mean(axis=1)          # share of resolutions each country voted on
    sub = sub[coverage >= min_coverage]
    S, C = agreement_matrix(sub)
    D = 1 - S                                    # distance = 1 - agreement
    D = D.fillna(D.stack().mean())               # fill rare gaps with the average distance
    Dm = D.to_numpy(copy=True)
    Dm = (Dm + Dm.T) / 2
    np.fill_diagonal(Dm, 0)
    Z = linkage(squareform(Dm, checks=False), method="average")
    labels = fcluster(Z, n_clusters, criterion="maxclust")
    return pd.Series(labels, index=D.index, name="cluster"), Z, S